In [27]:
import sys
sys.path.append(r'd:\VSCode\neylon-ai\primary-server')

In [57]:
pip install PyMuPDF

Note: you may need to restart the kernel to use updated packages.


In [30]:
import fitz

path = 'lib/data/Hruthik_m.pdf'
print(os.path.exists(path))

True


In [31]:
from django.core.files import File
from typing import List
from io import BytesIO
from PyPDF2 import PdfReader

RESUME_KEYWORDS = ["education", "experience", "skills", "projects", "linkedin", "email", "contact", "profile", "certifications", "React.js", "Next.js", "Tailwind CSS", "Langchain", "Langgraph", "Chroma DB", "OpenAI", "Google Gemini", "Django", "FastAPI", "Express.js", "Node.js", "Postman API", "WebSockets", "REST API", "REST APIs", "MongoDB", "PostgreSQL", "Google Cloud Platform", "GCP", "Firebase", "Python", "JavaScript", "C++", "Git", "GitHub", "Vercel", "Render", "VPS", "Figma", "Cloud Run", "Frontend", "Backend", "Fullstack", "Full-stack", "AI", "Machine Learning", "Deep Learning", "NLP"]

def is_resume(file: File, resume_words: List[str]):
    pdf_reader = PdfReader(BytesIO(file.read()))
    text = ""
    for page in pdf_reader.pages:
        text += page.extract_text() or ""
    text = text.lower()
    matches = sum(1 for word in resume_words if word in text)
    file.seek(0)
    return matches>=4

In [32]:
with open(path, "rb") as f:
    resume_file = File(f)
    print(is_resume(resume_file, RESUME_KEYWORDS))

True


In [33]:
import fitz  # PyMuPDF

# Open your PDF
doc = fitz.open("lib/data/Hruthik_M.pdf")

# Initialize containers
all_text = ""
all_links = set()

for page in doc:
    # Extract text
    all_text += page.get_text("text")

    # Extract links
    for link in page.get_links():
        if "uri" in link and link['uri'] not in all_links:  # external hyperlink
            all_links.add(link["uri"])   

print(f"Extracted {len(all_links)} unique links")

# Save text
with open("lib/data/output.txt", "w", encoding="utf-8") as f:
    f.write(all_text)

# Save links (optional)
with open("lib/data/links.txt", "w", encoding="utf-8") as f:
    for uri in sorted(all_links):
        f.write(uri + "\n")

print("✅ Text saved to output.txt")
print("🔗 Links extracted and saved to links.txt")

Extracted 8 unique links
✅ Text saved to output.txt
🔗 Links extracted and saved to links.txt


In [34]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
load_dotenv()

GEMINI_API_KEY=os.getenv("GOOGLE_API_KEY")

base_model = ChatGoogleGenerativeAI(
    model='gemini-2.5-flash',
    temperature=0.4,
    max_retries=2,
    google_api_key=GEMINI_API_KEY,
    streaming=True
)

In [55]:
from typing import Dict

def extract_resume(file: File) -> Dict[str, List[str]]:
    pdf_reader = PdfReader(BytesIO(file.read()))
    text = ""
    links = set()

    for page in pdf_reader.pages:
        text += page.extract_text() or ""
        annots = page.get("/Annots")
        if annots:
            for annot_ref in annots:
                annot = annot_ref.get_object()
                if annot.get("/A") and annot["/A"].get("/URI"):
                    links.add(annot["/A"]["/URI"])
    return {"text": text.strip(), "links": list(links)}

In [56]:
with open(path, "rb") as f:
    print(extract_resume(File(f)))

{'text': 'Hruthik M   \n+91 7483229386 |  mhrithik450@gmail.com  | LinkedI n | GitHu b    \n   \nExperience    \n  \n \nSoftware Development Engineer      April 2025 – Present     \nNext.js Developer  | Codedale             India   \n- Built and maintained 100+ robust backend endpoints  using TypeScript and Next.js, reducing API \nresponse time by 90% and improving system reliability for 100k users   \n- Designed and implemented PostgreSQL schemas and Drizzle ORM models  handling 20K+ \nrecords , optimizing queries and reducing average database load by 60%.  \n- Contributed to 50+ production features  end-to-end, including API design, authentication, and \ndata validation, supporting 10k+ active users .  \n \nFounder & Lead Engineer      Aug 2025 – Present     \nNext.js And Django Developer  | Neylon AI           India   \n- Founded Neylon AI , an AI agency delivering scalable AI assistants and agent -based solutions , \nserving diverse clients with intelligent automation.   \n- Develo

In [22]:
SYSTEM_PROMPT="""
You are an expert Resume Role Adaptation Assistant. Your goal is to intelligently tailor an existing resume to match a given job role or job description without losing structure, section order, or factual accuracy.

Rules:
0. Preserve Original Structure:
   - Keep the resume’s sections in the same order as the original, Keep all section titles exactly as they appear.
1. Selective Modification Based on Role:
   - Adjust the section lines to align with the provided job role. Only make modifications necessary to improve relevance.
   - Do not invent or remove any real experience, project, or education unless explicitly instructed by the user.
2. Preserve Authenticity:
   - Do not alter quantitative achievements or company names.
3. If Links are Present:
    - Retain all hyperlinks in their original positions. Replace each clickable text area with the link points to the same URL.
4. Integrate Additional Content if Provided:
   - If the user provides extra material, seamlessly insert it into the most relevant section — without breaking the structure.
5. Tone and Style:
   - Use action verbs and role-aligned keywords relevant to the target job description.
6. Output Format:
   - Return the entire modified resume text.
   - Ensure section headings, bullet and layout are clearly preserved. No explanations, comments, or notes — only the final formatted resume.

Generate a fully rewritten resume optimized for the target job role while preserving all original sections, order, and authenticity.
"""

In [23]:
import tiktoken 

encoding_model = "cl100k_base"
def get_encoding(text: str)->int:
    encoding = tiktoken.get_encoding(encoding_model)
    return len(encoding.encode(text))

In [24]:
print(get_encoding(SYSTEM_PROMPT))

293


In [25]:
user_prompt="""
Job Title: Full Stack Developer
Company: Cloudify Tech
Location: Remote

Description:
We are seeking a Full Stack Developer proficient in JavaScript and modern web frameworks. The ideal candidate will have hands-on experience developing scalable backend APIs, secure authentication systems, and interactive UIs.

Additional project:

CloudMetrics Dashboard
- Designed and developed a cloud monitoring dashboard using Next.js and Express.
- Integrated AWS CloudWatch APIs for real-time data visualization.
- Deployed application using Docker on AWS EC2.
"""

In [26]:
print(get_encoding(user_prompt))

101


In [35]:
from langchain.schema import HumanMessage, SystemMessage

messages = [
    SystemMessage(content=SYSTEM_PROMPT),
    HumanMessage(content=f"hyper_links: {all_links}\nold_resume: {all_text}\nuser: {user_prompt}")
]

In [ ]:
for chunk in base_model.stream(input=messages):
    print(chunk.content, end="", flush=True)

print("\n\n✅ Done streaming.")

In [17]:
with open("lib/data/response.txt", "w", encoding="utf-8") as f:
    f.write(gemini_response.content)

In [ ]:
print(gemini_response.content)

In [96]:
RESUME_EXTRACTOR_PROMPT = """
You are a highly accurate resume data extraction model.  
Your task is to read a raw resume text and extract all relevant information into a structured JSON object strictly following the schema below.

Rules:
- Always include all keys exactly as shown in the schema.
- If any section is missing or no content, clearly mention it with the string `"null"` (e.g., "<section_name>": {"title": "null","entries": []}).
- If any field is missing or not found, set its value to the string `"null"`.
- Return Only Valid JSON — no explanations, no comments, no extra text.
- Preserve lists, nested structures, and null placeholders properly.
- Extract multiple entries where applicable into lists as per below schema (e.g., multiple experiences, projects, education items).

Output format:
{{
    "name": "<person_name>",
    "contact": {{
        "phone": "<contact_number>",
        "email": "<email_address>",
        "links": {{
            "LinkedIn": "<linkedin_url>",
            "GitHub": "<github_url>",
            # optional: add more if needed
            "<key>": "<value>"
        }}
    }},
    "skills": {{
        "title": "<section_name>",
        # Example: "Skills"
        "entries": {{
            "<skill_category>": "<skills_list_comma_separated>",
            # Example: "Languages": "Python, JavaScript, C++"
        }}
    }},
    "experience": {{
        "title": "<section_name>",
        "entries": [
            {{
                "role": "<job_role>",
                "company": "<job_company>",
                "duration": "<job_duration>",
                # Example: "Aug 2017 – Aug 2020"
                "location": "<job_location>",
                "highlights": [
                    "<job_highlight_point>",
                    "..."
                ]
            }}
        ]
    }},
    "projects": {{
        "title": "<section_name>",
        "entries": [
            {{
                "name": "<project_name>",
                "duration": "<project_duration>",
                # Example: "Python, React.js, VPS"
                "links": {{
                    "Live": "<project_live_url>",
                    # optional
                    "<key>": "<value>"
                }},
                "highlights": [
                    "<project_highlight_point>",
                    "..."
                ]
            }}
        ]
    }},
    "certificates": {{
        "title": "<section_name>",
        "entries": [
            {{
                "name": "<certificate_name>",
                "duration": "<certificate_duration>",
                # Example: "March 2022"
                "links": {{
                    "CertificateLink": "<certificate_link>",
                    # optional
                    "<key>": "<value>"
                }},
                "highlights": [
                    "<certificate_highlight_point>",
                    "..."
                ]
            }}
        ]
    }},
    "education": {{
        "title": "<section_name>",
        "entries": [
            {{
                "institution": "<institution_name>",
                "cgpa": "<cgpa_value>",
                # Example: 8.0, 8.5
                "degree": "<degree_name>",
                # Example: "Bachelor of Technology in Mechanical Engineering"
                "duration": "<education_duration>",
                # Example: "Aug 2017 – Aug 2020"
                "location": "<education_location>"
            }}
        ]
    }}
}}
"""

In [97]:
print(get_encoding(RESUME_EXTRACTOR_PROMPT))

694


In [ ]:
import os
from langchain_openai import ChatOpenAI

OPENAI_API_KEY=os.getenv("OPENAI_API_KEY")
openai_model = ChatOpenAI(model="gpt-4o-mini", temperature=0.4, api_key=OPENAI_API_KEY)

In [99]:
openai_response = openai_model.invoke([
    {"role": "system", "content": RESUME_EXTRACTOR_PROMPT},
    {"role": "user", "content": gemini_response.content}
])

In [100]:
print(openai_response.content)

{
    "name": "Hruthik M",
    "contact": {
        "phone": "+91 7483229386",
        "email": "mhrithik450@gmail.com",
        "links": {
            "LinkedIn": "https://www.linkedin.com/in/hruthik-m-3595a0329?utm_source=share&utm_campaign=share_via&utm_content=profile&utm_medium=android_app",
            "GitHub": "https://github.com/Hrithik450"
        }
    },
    "skills": {
        "title": "Technical Skills",
        "entries": {
            "Frontend": "React.js, Next.js, Tailwind CSS",
            "AI Frameworks": "Langchain, Langgraph, Chroma DB, Open AI, Google Gemini",
            "Backend": "Django, FastAPI, Express.js, Node.js, Postman API, Websockets, REST API’s",
            "Databases": "MongoDB, PostgreSQL",
            "Cloud": "Google Cloud Platform, Firebase, AWS",
            "Languages": "Python, JavaScript, C++",
            "Tools": "Git, GitHub, Vercel, Render, VPS, Figma, Cloud Run, Docker"
        }
    },
    "experience": {
        "title": "Experience",

In [24]:
import re
import json

def parse_json(raw_response):
    if not raw_response:
        return None
    match = re.search(r'\{.*\}', raw_response, re.S)
    if match:
        return json.loads(match.group(0))
    return None

In [ ]:
resume_data = parse_json(openai_response.content)
print(test_test_test_resume_data)

{'name': 'Hruthik M', 'contact': {'phone': '+91 7483229386', 'email': 'mhrithik450@gmail.com', 'links': {'LinkedIn': 'https://www.linkedin.com/in/hruthik-m-3595a0329?utm_source=share&utm_campaign=share_via&utm_content=profile&utm_medium=android_app', 'GitHub': 'https://github.com/Hrithik450'}}, 'skills': {'title': 'Technical Skills', 'entries': {'Frontend': 'React.js, Next.js, Tailwind CSS', 'AI Frameworks': 'Langchain, Langgraph, Chroma DB, Open AI, Google Gemini', 'Backend': 'Django, FastAPI, Express.js, Node.js, Postman API, Websockets, REST API’s', 'Databases': 'MongoDB, PostgreSQL', 'Cloud': 'Google Cloud Platform, Firebase, AWS (CloudWatch, EC2)', 'Languages': 'Python, JavaScript, C++', 'Tools': 'Git, GitHub, Vercel, Render, VPS, Figma, Cloud Run, Docker'}}, 'experience': {'title': 'Experience', 'entries': [{'role': 'Software Development Engineer', 'company': 'Codedale', 'duration': 'April 2025 – Present', 'location': 'India', 'highlights': ['Built and maintained 100+ robust back

In [28]:
print(resume_data)
print(get_encoding(openai_response.content))

{'name': 'Hruthik M', 'contact': {'phone': '+91 7483229386', 'email': 'mhrithik450@gmail.com', 'links': {'LinkedIn': 'https://www.linkedin.com/in/hruthik-m-3595a0329?utm_source=share&utm_campaign=share_via&utm_content=profile&utm_medium=android_app', 'GitHub': 'https://github.com/Hrithik450'}}, 'skills': {'title': 'Technical Skills', 'entries': {'Frontend': 'React.js, Next.js, Tailwind CSS', 'AI Frameworks': 'Langchain, Langgraph, Chroma DB, Open AI, Google Gemini', 'Backend': 'Django, FastAPI, Express.js, Node.js, Postman API, Websockets, REST API’s', 'Databases': 'MongoDB, PostgreSQL', 'Cloud': 'Google Cloud Platform, Firebase, AWS (CloudWatch, EC2)', 'Languages': 'Python, JavaScript, C++', 'Tools': 'Git, GitHub, Vercel, Render, VPS, Figma, Cloud Run, Docker'}}, 'experience': {'title': 'Experience', 'entries': [{'role': 'Software Development Engineer', 'company': 'Codedale', 'duration': 'April 2025 – Present', 'location': 'India', 'highlights': ['Built and maintained 100+ robust back

In [78]:
import os

BASE_DIR = os.getcwd()
print(BASE_DIR)

d:\VSCode\neylon-ai\primary-server\resume_assistant


In [5]:
from reportlab.lib.pagesizes import A4
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.platypus import Paragraph, SimpleDocTemplate, Spacer, Table, TableStyle, HRFlowable
from lib.utils import test_resume_data
import os

BASE_DIR = os.getcwd()
print(BASE_DIR)

pdfmetrics.registerFont(TTFont('Guminert-Regular', os.path.join(BASE_DIR, "assets/fonts/Guminert-Regular.ttf")))
pdfmetrics.registerFont(TTFont('Guminert-Medium', os.path.join(BASE_DIR, "assets/fonts/Guminert-Medium.ttf")))

pdf_file = "lib/data/Hruthik_M_Genrated_Resume.pdf"
doc = SimpleDocTemplate(pdf_file, pagesize=A4, rightMargin=10, leftMargin=10, topMargin=2, bottomMargin=2)

styles = getSampleStyleSheet()
normal = styles['Normal']
story = []

# Name
story.append(Paragraph(test_resume_data["name"], ParagraphStyle(name="Name", fontSize=28, alignment=1, spaceAfter=8, leading=28, fontName="Guminert-Medium")))

# Contact
contact = test_resume_data.get("contact", {})
links = contact.get("links", {})

# Extract safely with fallbacks
phone = contact.get("phone", "null")
email = contact.get("email", "null")
linkedin = links.get("LinkedIn", "null")
github = links.get("GitHub", "null")

contact_parts = []
if phone != "null":
    contact_parts.append(f"{phone}")
if email != "null":
    contact_parts.append(f"<link href='mailto:{email}' color='#085A8C' underline='true'>{email}</link>")
if linkedin != "null":
    contact_parts.append(f"<link href='{linkedin}' color='#085A8C' underline='true'>{linkedin}</link>")
if github != "null":
    contact_parts.append(f"<link href='{github}' color='#085A8C' underline='true'>{github}</link>")

if len(contact_parts) > 3:
    contact_info = ("&nbsp;&nbsp;".join(contact_parts[:3]) + "<br/>" + "&nbsp;&nbsp;".join(contact_parts[3:]))
else:
    contact_info = "&nbsp;&nbsp;".join(contact_parts)
story.append(Paragraph(contact_info, ParagraphStyle(name="Contact", fontSize=12, alignment=1, spaceAfter=12, leading=17, fontName="Guminert-Regular")))

# Education Section
if test_resume_data.get("education") and test_resume_data["education"].get("title") != "null" and test_resume_data["education"].get("entries"):
    story.append(Paragraph(f'<b>{test_resume_data["education"]["title"]}</b>', ParagraphStyle(name="education", fontSize=13, leading=16, spaceBefore=8, underlineWidth=1, fontName="Guminert-Medium")))
    story.append(HRFlowable(width="100%", thickness=0.5, lineCap='round', color="#000000", spaceBefore=2, spaceAfter=2))

    for entry in test_resume_data["education"]["entries"]:
        institution = entry.get("institution", "null")
        duration = entry.get("duration", "null")
        degree = entry.get("degree", "null")
        location = entry.get("location", "null")
        cgpa = entry.get("cgpa", "null")

        row1 = []
        row2 = []

        if cgpa != "null" and institution != "null":
            row1.append(Paragraph(f'<b>{institution}, G.P.A: {cgpa}</b>', ParagraphStyle(name="institution_and_gpa", parent=normal, fontName="Guminert-Regular", fontSize=11)))
        
        if cgpa == "null" and institution != "null":
            row1.append(Paragraph(f'<b>{institution}</b>', ParagraphStyle(name="institution", parent=normal, fontName="Guminert-Regular", fontSize=12))) if institution != "null" else row1.append("")

        if duration != "null":
            row1.append(Paragraph(f'<b>{duration}</b>', ParagraphStyle(name="duration", parent=normal, fontName="Guminert-Medium", fontSize=12, alignment=2)))

        if degree != "null":
            row2.append(Paragraph(f'{degree}', ParagraphStyle(name="degree", parent=normal, fontName="Guminert-Regular", fontSize=12)))
        
        if location != "null":
            row2.append(Paragraph(f'{location}', ParagraphStyle(name="location", parent=normal, fontName="Guminert-Regular", fontSize=12, alignment=2)))

        data = [row1, row2]
        table = Table(data, colWidths=["75%", "25%"])
        table.setStyle(TableStyle([
            ("VALIGN", (0, 0), (-1, -1), "TOP"),
            ("ALIGN", (1, 0), (1, 0), "RIGHT"),
            ("BOTTOMPADDING", (0, 0), (-1, -1), 5),
            ("TOPPADDING", (0, 0), (-1, -1), 0),
            ("LEFTPADDING", (0, 0), (-1, -1), 0),
            ("RIGHTPADDING", (0, 0), (-1, -1), 0)
        ]))
        story.append(table)

# Skills Section
if test_resume_data.get("skills") and test_resume_data["skills"].get("title") != "null" and test_resume_data["skills"].get("entries"):
    story.append(Paragraph(f'<b>{test_resume_data["skills"]["title"]}</b>', ParagraphStyle(name="section_title", fontSize=13, leading=16, spaceBefore=8, underlineWidth=1, fontName="Guminert-Medium")))
    story.append(HRFlowable(width="100%", thickness=0.5, lineCap='round', color="#000000", spaceBefore=2, spaceAfter=2))

    for key, value in test_resume_data["skills"]["entries"].items():
        story.append(Paragraph(f"<b>{key}:</b> {value}", ParagraphStyle(name="skill_points", fontSize=12, spaceBefore=2, spaceAfter=2, bulletIndent=10, leading=15, fontName="Guminert-Regular")))

# Experience Section
if test_resume_data.get("experience") and test_resume_data["experience"].get("title") != "null" and test_resume_data["experience"].get("entries"):
    story.append(Paragraph(f'<b>{test_resume_data["experience"]["title"]}</b>', ParagraphStyle(name="section_title", fontSize=13, leading=16, spaceBefore=8, underlineWidth=1, fontName="Guminert-Medium")))
    story.append(HRFlowable(width="100%", thickness=0.5, lineCap='round', color="#000000", spaceBefore=2, spaceAfter=2))

    for entry in test_resume_data.get("experience", {}).get("entries", []):
        role = entry.get("role", "null")
        company = entry.get("company", "null")
        duration = entry.get("duration", "null")
        location = entry.get("location", "null")
        highlights = entry.get("highlights", [])

        if role != "null":
            job_role = Paragraph(f'<b>{role}</b>', ParagraphStyle(name="job_role", parent=normal, fontName="Guminert-Medium", fontSize=12))

        if company != "null":
            job_company = Paragraph(company, ParagraphStyle(name="job_company", parent=normal, fontName="Guminert-Regular", fontSize=12))

        if duration != "null":
            job_duration = Paragraph(f'<b>{duration}</b>', ParagraphStyle(name="job_duration", parent=normal, fontName="Guminert-Medium", fontSize=12, alignment=2))

        if location != "null":
            job_location = Paragraph(location, ParagraphStyle(name="job_location", parent=normal, fontName="Guminert-Regular", fontSize=12, alignment=2))

        if location != "null" and duration != "null" and company != "null" and role != "null":
            data = [[job_role, job_duration], [job_company, job_location]]
        elif duration != "null" and company != "null" and role != "null":
            data = [[job_role, job_duration], [job_company, ""]]
        elif location != "null" and company != "null" and role != "null":
            data = [[job_role, ""], [job_company, job_location]]
        elif location != "null" and duration != "null" and role != "null":
            data = [[job_role, job_duration], ["", job_location]]
        elif company != "null" and role != "null":
            data = [[job_role, ""], [job_company, ""]]
        elif location != "null" and role != "null":
            data = [[job_role, ""], ["", job_location]]
        elif duration != "null" and role != "null":
            data = [[job_role, job_duration], ["", ""]]
        else:
            data = [[job_role, ""], ["", ""]]

        table = Table(data, colWidths=["70%", "30%"])
        table.setStyle(TableStyle([
            ("VALIGN", (0, 0), (-1, -1), "TOP"),
            ("ALIGN", (1, 0), (1, 0), "RIGHT"),
            ("BOTTOMPADDING", (0, 0), (-1, -1), 4),
            ("TOPPADDING", (0, 0), (-1, -1), 0),
            ("LEFTPADDING", (0, 0), (-1, -1), 0),
            ("RIGHTPADDING", (0, 0), (-1, -1), 0)
        ]))
        story.append(table)

        if highlights:
            for point in highlights:
                story.append(Paragraph(f"<bullet>&bull;</bullet> {point}", ParagraphStyle(name="bullet_points", fontSize=12, leftIndent=20, spaceBefore=2, spaceAfter=2, bulletIndent=10, leading=16, fontName="Guminert-Regular")))
        story.append(Spacer(1, 4))

# Projects Section
if test_resume_data.get("projects") and test_resume_data["projects"].get("title") != "null" and test_resume_data["projects"].get("entries"):
    story.append(Paragraph(f'<b>{test_resume_data["projects"]["title"]}</b>', ParagraphStyle(name="section_title", fontSize=13, leading=16, spaceBefore=8, underlineWidth=1, fontName="Guminert-Medium")))
    story.append(HRFlowable(width="100%", thickness=0.5, lineCap='round', color="#000000", spaceBefore=2, spaceAfter=2))

    for entry in test_resume_data["projects"]["entries"]:
        name = entry["name"]
        duration = entry.get("duration", "null")
        live_link = entry.get("links", {}).get("Live", "null")

        if live_link != "null":
            project_name = Paragraph(f"<b>{name}</b> | <link href='{live_link}' color='#085A8C' underline='true'>View Live</link>", ParagraphStyle(name="ProjectName", parent=normal, fontName="Guminert-Medium", fontSize=12))
        else:
            project_name = Paragraph(f"<b>{name}</b>", ParagraphStyle(name="ProjectName", parent=normal, fontName="Guminert-Medium", fontSize=12))

        if duration != "null":
            duration_para = Paragraph(duration, ParagraphStyle(name="Duration", parent=normal, fontName="Guminert-Medium", fontSize=12, alignment=2))
        
        if duration != "null":
            data = [[project_name, duration_para]]
        else:
            data = [[project_name, ""]]
    
        table = Table(data, colWidths=["70%", "30%"])
        table.setStyle(TableStyle([
            ("VALIGN", (0, 0), (-1, -1), "TOP"),
            ("ALIGN", (1, 0), (1, 0), "RIGHT"),
            ("BOTTOMPADDING", (0, 0), (-1, -1), 4),
            ("TOPPADDING", (0, 0), (-1, -1), 0),
            ("LEFTPADDING", (0, 0), (-1, -1), 0),
            ("RIGHTPADDING", (0, 0), (-1, -1), 0)
        ]))
        story.append(table)

        for point in entry["highlights"]:
            story.append(Paragraph(f"<bullet>&bull;</bullet> {point}", ParagraphStyle(name="BulletPoints", fontSize=12, leftIndent=20, spaceBefore=2, spaceAfter=2, bulletIndent=10, leading=16, fontName="Guminert-Regular")))
        story.append(Spacer(1, 4))

# Certificates Section
if test_resume_data.get("certificates") and test_resume_data["certificates"].get("title") != "null" and test_resume_data["certificates"].get("entries"):
    story.append(Paragraph(f'<b>{test_resume_data["certificates"]["title"]}</b>', ParagraphStyle(name="SectionTitle", fontSize=13, leading=16, spaceBefore=8, underlineWidth=1, fontName="Guminert-Medium")))
    story.append(HRFlowable(width="100%", thickness=0.5, lineCap='round', color="#000000", spaceBefore=2, spaceAfter=2))

    for entry in test_resume_data["certificates"]["entries"]:
        name = entry["name"]
        duration = entry.get("duration", "")
        highlights = entry.get("highlights", [])
        certificate_link = entry.get("links", {}).get("CertificateLink")

        if certificate_link:
            certificate_name = Paragraph(f"<b>{name}</b> | <link href='{certificate_link}' color='#085A8C' underline='true'>Certificate</link>", ParagraphStyle(name="CertificateName", parent=normal, fontName="Guminert-Regular", fontSize=12))
        else:
            certificate_name = Paragraph(f"<b>{name}</b>", ParagraphStyle(name="CertificateName", parent=normal, fontName="Guminert-Regular", fontSize=12))

        if duration != "null":
            duration_para = Paragraph(duration, ParagraphStyle(name="Duration", parent=normal, fontName="Guminert-Regular", fontSize=12, alignment=2))
            data = [[certificate_name, duration_para]]
        else:
            data = [[certificate_name, ""]]

        table = Table(data, colWidths=["75%", "25%"])
        table.setStyle(TableStyle([
            ("VALIGN", (0, 0), (-1, -1), "TOP"),
            ("ALIGN", (1, 0), (1, 0), "RIGHT"),
            ("BOTTOMPADDING", (0, 0), (-1, -1), 4),
            ("TOPPADDING", (0, 0), (-1, -1), 0),
            ("LEFTPADDING", (0, 0), (-1, -1), 0),
            ("RIGHTPADDING", (0, 0), (-1, -1), 0)
        ]))
        story.append(table)

        if highlights:
            for point in highlights:
                story.append(Paragraph(f"<bullet>&bull;</bullet> {point}", ParagraphStyle(name="BulletPoints", fontSize=12, leftIndent=20, spaceBefore=2, spaceAfter=2, bulletIndent=10, leading=16, fontName="Guminert-Regular")))

doc.build(story)
print(f"✅ Resume saved as {pdf_file}")

d:\VSCode\neylon-ai\primary-server\resume_assistant
✅ Resume saved as lib/data/Hruthik_M_Genrated_Resume.pdf


In [62]:
import os
import json
import base64
from google.oauth2 import service_account
from dotenv import load_dotenv

load_dotenv()
creds_bs64 = os.getenv("GOOGLE_CREDENTIALS_BASE64")
if creds_bs64:
    creds_json = json.loads(base64.b64decode(creds_bs64))
    GS_CREDENTIALS = service_account.Credentials.from_service_account_info(creds_json)
else:
    GS_CREDENTIALS = None

print(GS_CREDENTIALS)